<a href="https://colab.research.google.com/github/hayannn/AIFFEL_STUDY/blob/main/4_7_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07. LightGBM
- ?? GBM ??? ?? ?? ??? ?? ?? ?? ?? ??(Leaf Wise) ?? ??

<br>

- ???? ?? ?? ????
  - ?? ?? ??(Level Wise) ?? ?? : ??? ?? ?? ??? ????? ??? ??? ??? ???? ? ??
  - ??? ??? ?? ??? ????? ?? ??

<br>

- LightGBM? ?? ?? ?? ?? ??
  - ?? ??? ??? ??, ?? ?? ?(max delta loss)? ??? ?? ??? ????? ????? ??? ??? ???? ????? ?? ?? ??
  - ?? ???? ??? ?? ??? ????? ??? ??? ?? ??? ??? ????? ??? ?? ?? ?? ???? ?? ?? ??? ???? ? ??? ?

<br>

- ??
  - ? ?? ??? ?? ?? ??
  - ? ?? ??? ???
  - ????? ?? ?? ?? ? ?? ??

In [ ]:
import lightgbm

print(lightgbm.__version__)

### LightGBM ??? ????
> ?? : Xgboost? ??? ?? ??? ?? ????? ??? ??? ????? ??? ?? ??? ?? ??? ???? ??? ??

- ?? ????
  - `num_iterations [defalut = 100]`
    - ?? ????? ?? ?? ??
    - ?? ?? -> ?? ?? ?????(?? ?? ?? ? ??? ?? ?? ??)
    - ???? ?? ??? -> n_estimators? ?? ??
  -  `learning_rate [default = 0.1]`
    - 0~1 ?? ? ??, ??? ??? ????? ??? ? ?????? ??? ?
    - n_estimators? ?? ?? learning_rate? ?? ?? ?? ??? ???? ? ???, ??? ??, ?? ??? ???? ??? ?? ?? ??
  - `max_depth [default=-1]`
    - 0?? ?? ? ???? ??? ?? ??
    - Depth wise ??? ?? ??? ??, Leaf wise ????? ??? ????? ? ??
  - `min_data_in_leaf [default=20]`
    - ?? ??? min_samples_leaf? ?? ????
    - ???? ??? LightGBMClassifier? min_child_samples ????? ?? ??
    - ?? ?? ???? ?? ??? ?? ?? ????? ??? ??? ?
    - ??? ??? ?? ????
  - `num_leaves [default=31]`
    - ??? ??? ?? ? ?? ?? ?? ??
  - `boosting [default=gbdt]`
    - ??? ??? ???? ???? ??
    - gbdt : ???? ????? ??? ?? ??
    - rf : ?? ????
  - `bagging_fraction [default = 1.0]`
    - ??? ?? ????? ?? ???? ?? ???? ????? ??? ??
    - ????? GBM? XGBClassifier? subsample ????? ??
  - `feature_fraction [default=1.0]`
    - ?? ?? ?? ??? ???? ???? ?? ??(??? ??? ?? ??)
  - `lambda_l2 [defalut=0.0]`
    - L2 regulation ??
    - ?? ??? ?? ?? ?? ??, ?? ??? ??? ?? ??? ??
  - `lambda_l1 [default=0.0]`
    - L1 regulation ??
    - L2? ????? ??? ??

<br>

- Learing Task ????
  - objective : ???? ??? ? ???? ??
  - xgboost? objective ????? ??
  - ?????? ??, ? ??, ?? ??? ??, ?? ????? ?? objective? ???? ??

### ??? ???? ?? ??
- **num_leaves ??? ???? min_child_samples(min_data_in_leaf), max_depth? ?? ??? ?? ???? ??? ?? ?? ?? ??**
- learning_rate? ?? ?? n_estimators? ?? ?? ?? ??? ?? ???? ?? ???? ?? ??(?, n_estimators? ?? ?? ?? ????? ??? ?? ??)
- ??? ??
  - reg_lambda, reg_alpha? ?? regularization ??
  - ?? ???? ??? ?? ??? ??? ??? ??? ??? ??? ?? colsample_brtree, subsample ???? ??

### ??? ?? LightGBM vs ???? ?? XGBoost vs LightGBM ??? ????

| ??? ?? LightGBM | ???? ?? LightGBM | ???? ?? XGBoost |
|-------------------|--------------------|---------------------|
| num_iterations | n_estimators | n_estimators |
| learning_rate | learning_rate | learning_rate |
| max_depth | max_depth | max_depth |
| min_data_in_leaf | min_child_samples | N/A |
| bagging_fraction | subsample | subsample |
| feature_fraction | colsample_bytree | colsample_bytree |
| lambda_l2 | reg_lambda | reg_lambda |
| lambda_l1 | reg_aplha | reg_alpha |
| early_stopping_round | early_stopping_rounds | early_stopping_rounds |
| num_leaves | num_leaves | N/A |
| min_sum_hessian_in_leaf | min_child_weight | min_child_weight |

### LightGBM ?? ? ???? Breast Cancer Prediction

[lightgbm ???? ???? ??](https://www.inflearn.com/community/questions/1134636/lightgbm-%EC%A1%B0%EA%B8%B0%EC%A4%91%EB%8B%A8-%ED%8C%8C%EB%9D%BC%EB%AF%B8%ED%84%B0-%EC%98%A4%EB%A5%98?srsltid=AfmBOop5zJqwJ_GywXqqmuoEgTtA86V8ylLOqoLPSdyKbO4T5QdKJhGq)

In [ ]:
# LightGBM? ??? ???? lightgbm?? LGBMClassifier ???
from lightgbm import LGBMClassifier

import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from lightgbm import early_stopping

dataset = load_breast_cancer()

cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']= dataset.target
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

# ?? ??? ? 80%? ??? ???, 20%? ???? ??? ??
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

# ??? ?? X_train, y_train? ?? ??? 90%? ??? 10%? ??? ???? ??
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

# ?? XGBoost? ???? n_estimators? 400 ??
lgbm_wrapper = LGBMClassifier(n_estimators=400, learning_rate=0.05)

# LightGBM? XGBoost? ???? ?? ?? ?? ??
evals = [(X_tr, y_tr), (X_val, y_val)]
lgbm_wrapper.fit(X_tr, y_tr, callbacks=[early_stopping(stopping_rounds=50)], eval_metric="logloss", eval_set=evals)
preds = lgbm_wrapper.predict(X_test)
pred_proba = lgbm_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC ??
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('?? ??')
    print(confusion)
    # ROC-AUC print ??
    print('???: {0:.4f}, ???: {1:.4f}, ???: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
get_clf_eval(y_test, preds, pred_proba)

In [ ]:
# plot_importance( )? ???? feature ??? ???
from lightgbm import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(lgbm_wrapper, ax=ax)
plt.savefig('lightgbm_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')